# 02 - Compute and Write a Transient Parameter

Read `vCar`, derive `vCar-50`, write it back to the session as a transient (calculated) parameter, and show both channels on a waveform display in ATLAS.

**Before you run:** start ATLAS 11+ and load a session containing `vCar:Chassis` into Set 1.

This notebook is idempotent - re-run it freely; the previous transient is removed first.

In [ ]:
import os
from pythonnet import load

ATLAS_DIR = os.environ.get(
    "ATLAS_DIR", r"C:\Program Files\McLaren Applied Technologies\ATLAS 10"
)
load("coreclr", runtime_config=os.path.join(ATLAS_DIR, "MAT.Atlas.Host.runtimeconfig.json"))

import clr
clr.AddReference(os.path.join(ATLAS_DIR, "MAT.Atlas.Automation.Api.dll"))
clr.AddReference(os.path.join(ATLAS_DIR, "MAT.Atlas.Automation.Client.dll"))

from MAT.Atlas.Automation.Api.Enums import DataStatusType
from MAT.Atlas.Automation.Client.Services import (
    DisplayServiceClient,
    ParameterDataAccessServiceClient,
    SessionServiceClient,
    SetServiceClient,
    WorkbookServiceClient,
)

SOURCE_PARAMETER = "vCar:Chassis"
TRANSIENT_ID = "vCarMinus50:Demo"
DISPLAY_NAME = "Automation Demo"
OFFSET = 50.0

workbook = WorkbookServiceClient()
sets = SetServiceClient()
session_service = SessionServiceClient()
pda = ParameterDataAccessServiceClient()
displays = DisplayServiceClient()
print("Ready.")

## Read vCar and derive vCar-50

In [ ]:
session_id = sets.GetCompositeSessions(workbook.GetSets()[0].Id)[0].Id
parameter = session_service.GetSessionParameter(session_id, SOURCE_PARAMETER)
timebase = session_service.GetSessionTimeBase(session_id)
estimate = pda.GetSamplesCountEstimate(session_id, parameter.Identifier, timebase.StartTime, timebase.EndTime)

pda.Goto(session_id, parameter.Identifier, timebase.StartTime)
samples = pda.GetNextSamples(session_id, parameter.Identifier, estimate)

timestamps, values = [], []
for i in range(samples.SampleCount):
    if samples.DataStatus[i].HasFlag(DataStatusType.Sample):
        timestamps.append(samples.Time[i])
        values.append(samples.Data[i] - OFFSET)

print(f"Derived {len(values)} samples of {SOURCE_PARAMETER} - {OFFSET}.")

## Write the transient parameter

In [ ]:
try:
    session_service.RemoveTransientParameter(session_id, TRANSIENT_ID)
except Exception:
    pass  # not present yet

transient = session_service.AddTransientParameter(
    session_id, TRANSIENT_ID, "vCarMinus50", "vCar offset by 50 km/h", ["Demo"], 0, 800
)
session_service.AddTimeDataToTransientParameter(session_id, transient.Identifier, timestamps, values)
print(f"Wrote {len(values)} samples to {transient.Identifier}.")

## Show both channels on a waveform display

In [ ]:
display_id = next((d.Id for d in workbook.GetDisplays() if d.Name == DISPLAY_NAME), None)
if display_id is None:
    display_id = workbook.CreateDisplay("Waveform", DISPLAY_NAME).Id

on_display = {p.Identifier for p in displays.GetDisplayParameters(display_id)}
for identifier in (parameter.Identifier, transient.Identifier):
    if identifier not in on_display:
        displays.AddDisplayParameter(display_id, identifier)

print(f"Display '{DISPLAY_NAME}' now shows {parameter.Identifier} and {transient.Identifier}.")
print("Switch to ATLAS to see the waveform.")

In [ ]:
for client in (workbook, sets, session_service, pda, displays):
    client.Dispose()